# Lesson 12 — Reaction-Rate Tallies, $\bar{\eta}$, and  $k_{\infty}$

Building on the previous lessons, we will

1. Add explicit `nu-fission` and `absorption` tallies and divide by flux to obtain
   spectrum-weighted $\overline{\nu\Sigma_f}$ and $\overline{\Sigma_a}$.
2. Define the production/absorption ratio with its **absorption denominator stated**.
3. Sweep the H/U **atom** ratio, locate a sampled moderation optimum, and check
   its magnitude against the anticipated roughly 0.85 scale.
4. Turn the external-source calculation into a $k$-eigenvalue calculation and
   see why the two spectra and the two ratios are not automatically identical.

**Draft for instructor review.** Select a Python kernel containing OpenMC,
NumPy, pandas, and Matplotlib. The OpenMC executable and a continuous-energy
neutron-data library, including `c_H_in_H2O`, must be available. Activate your
OpenMC environment before launching Jupyter; its `OPENMC_CROSS_SECTIONS`
variable should name `cross_sections.xml`.

This notebook is standalone: it does not execute or modify another lesson.
Runs write XML/HDF5 files only to the scratch directory printed below.
The density is the same pedagogical **1 g/cm³** as Lesson 10; this is a
homogeneous uranium–water mixture, not a heterogeneous fuel lattice or a
chemical-solubility model. Oxygen here belongs to water, not added UO₂.

In [1]:
import openmc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# GLOBAL CONSTANTS
TEMPERATURE = 294.0
RADIUS = 1.0 # cm, same reflective sphere as Lesson 10
VOLUME = 4.0 * np.pi * RADIUS**3 / 3.0
K_B = 8.617333262e-5  # eV/K
ENERGY_EDGES = np.geomspace(1e-5, 2e7, 1001)  # geomspace like logspace but with absolute points
ENERGY = np.sqrt(ENERGY_EDGES[:-1] * ENERGY_EDGES[1:]) # "center energy"
DELTA_E = np.diff(ENERGY_EDGES)
DELTA_U = np.log(ENERGY_EDGES[1:] / ENERGY_EDGES[:-1])

PURPLE="#512888"
ORANGE="#CA7C1B"

## 1. Tally reaction rates first; derive effective cross sections second

For the cell-integrated spectrum, define

$$\Phi=\int_V\!\int_0^\infty\phi(\mathbf r,E)\,dE\,dV,\qquad
 P=\int_V\!\int_0^\infty\nu\Sigma_f(E)\phi(\mathbf r,E)\,dE\,dV,$$
$$A=\int_V\!\int_0^\infty\Sigma_{a}(E)\phi(\mathbf r,E)\,dE\,dV.$$

Then

$$
\begin{split}
\overline{\nu\Sigma_f}         &= P/\Phi \\
\overline{\Sigma_{a}}  &= A/\Phi \\
\overline{\eta}                &= P/A \, .
\end{split}
$$

OpenMC `nu-fission` and `absorption` are **reaction-rate scores**, not direct
cross sections. Dividing either by the corresponding `flux` tally yields an
effective macroscopic cross section in cm$^{-1}$.  

## 2. Keep the model; make fission-source handling explicit

Here **fixed source** means that each history starts with the specified 1 MeV
external neutron. For the main sweep, `create_fission_neutrons=True`: fission
descendants are also transported. They are sampled from fission data, not
forced to 1 MeV. This is an externally driven, multiplying, subcritical system.

We also repeat one case with descendants suppressed, as in Lesson 11, to
isolate the importance of this choice. Do not run an unbounded multiplying
fixed-source history after changing the material to a supercritical one;
use eigenvalue mode to study that case. The natural-U/water cases below are
checked against a subcritical eigenvalue result.

In [2]:
def make_model(H_to_U=100.0, temperature=294, bound_hydrogen=True,
               multiplying=False, particles=2000, batches=40, seed=11001):
    """Lesson 11's model with extra tallies.
    """
    material = openmc.Material(name="natural U + water")
    
    # Bypass U with H_to_U set to None
    N_U = 1.0
    if H_to_U is None:
        H_to_U = 2
        N_U = 0
        
    material.add_nuclide("H1", float(H_to_U))
    material.add_nuclide("O16", float(H_to_U) / 2.0)
    material.add_element("U", N_U)  # Use OpenMC's natural isotopic abundances
    material.set_density("g/cm3", 1.0)
    material.temperature = float(temperature)
    if bound_hydrogen:
        material.add_s_alpha_beta("c_H_in_H2O")

    sphere = openmc.Sphere(r=RADIUS, boundary_type="reflective")
    
    cell = openmc.Cell(fill=material, region=-sphere)
    
    settings = openmc.Settings()
    settings.run_mode = "fixed source"
    settings.source = openmc.IndependentSource(
        space=openmc.stats.Point((0.0, 0.0, 0.0)),
        energy=openmc.stats.Discrete([2e6], [1.0]),
    )
    settings.particles = int(particles)
    settings.batches = int(batches)
    settings.seed = int(seed)
    settings.create_fission_neutrons = bool(multiplying)
    
    materials = openmc.Materials([material])
    
    model = openmc.Model(geometry=openmc.Geometry([cell]), materials=materials, settings=settings)
    
    cell_filter = openmc.CellFilter(cell)
    spectrum = openmc.Tally(name="spectrum")
    spectrum.filters = [cell_filter, openmc.EnergyFilter(ENERGY_EDGES)]
    spectrum.scores = ["flux"]
    spectrum.estimator = "tracklength"
    
    integral = openmc.Tally(name="integral reaction rates")
    integral.filters = [cell_filter]  # no EnergyFilter: include all transported energies
    integral.scores = ["flux", "nu-fission", "absorption"]
    integral.estimator = "tracklength"
    
    model.tallies = openmc.Tallies([spectrum, integral])
    
    return model, material, cell

In [3]:
def get_results(sp_path):
    with openmc.StatePoint(sp_path) as sp:
        spectrum = sp.get_tally(name="spectrum")
        mean = spectrum.get_values(scores=["flux"], value="mean").ravel().copy()
        sd = spectrum.get_values(scores=["flux"], value="std_dev").ravel().copy()
        results = {"bin_flux": mean, "bin_sd": sd,
                   "phi_E": mean / (VOLUME * DELTA_E),
                   "phi_u": mean / (VOLUME * DELTA_U)}
        
        integral = sp.get_tally(name="integral reaction rates")
        means = {score: float(integral.get_values(scores=[score], value="mean").sum())
                     for score in integral.scores}
        Phi, P, A = (means[score] for score in ["flux", "nu-fission", "absorption"])
        
        results["Phi"] = means["flux"]
        results["P"] = means["nu-fission"]
        results["A"] = means["absorption"]
        
        return results

## 4. Exploring $\bar{\eta}$ with $H/U$


In [4]:
for HU in [100, 50, 25, 20, 15, 10, 5, 4, 3]:
    model, _, _ = make_model(H_to_U=HU, bound_hydrogen=True, multiplying=True)
    sp_path = model.run(output=False)
    results = get_results(sp_path)
    print(f"H/U={HU}    η = {results['P']/results['A']}")

H/U=100    η = 0.2377704680402101
H/U=50    η = 0.3920635085738491
H/U=25    η = 0.5795378378136462
H/U=20    η = 0.6392727619812562
H/U=15    η = 0.7112613822229615
H/U=10    η = 0.7951676323345119
H/U=5    η = 0.8795770566193365
H/U=4    η = 0.8880329352954883
H/U=3    η = 0.8877435413186401
